# Clasificadores en Aprendizaje Automático

Este notebook combina **teoría**, **ejemplos prácticos**, **visualizaciones** y **ejercicios** para aprender cómo funcionan los **clasificadores** en Machine Learning con `scikit-learn`.

---
## Objetivos de aprendizaje
- Comprender el flujo de trabajo de un clasificador supervisado.
- Entrenar, evaluar y comparar varios clasificadores clásicos.
- Interpretar métricas de evaluación (accuracy,
precision, recall, F1, ROC-AUC) y la matriz de confusión.
- Visualizar fronteras de decisión en 2D.
- Realizar experimentos guiados con ejercicios para consolidar el aprendizaje.



## Introducción teórica
Un **clasificador** es un modelo de **aprendizaje supervisado** que, dada una instancia con variables de entrada **X**, intenta predecir su **clase** (etiqueta) **y**.  
Aprende a partir de un conjunto de ejemplos etiquetados (datos de entrenamiento).

### Tipos de problemas de clasificación
- **Binaria:** dos clases (p. ej., spam / no spam).
- **Multiclase:** tres o más clases (p. ej., Iris: setosa, versicolor, virginica).


### Flujo general de trabajo
1) **Preparación de datos:** limpieza, selección de variables, escalado si aplica.  
2) **Partición:** entrenamiento / validación / prueba.  
3) **Entrenamiento:** ajuste de parámetros del modelo minimizando una función de pérdida.  
4) **Predicción:** el modelo estima la clase de nuevas instancias.  
5) **Evaluación:** se usan métricas apropiadas según el problema.  
6) **Ajuste de hiperparámetros:** búsqueda de mejores configuraciones (grid/random/bayes).  
7) **Despliegue y monitoreo:** uso en producción y seguimiento del rendimiento.

### Clasificadores clásicos (resumen)
- **Regresión Logística (LogisticRegression):** lineal, produce probabilidades; buena línea base.  
- **k Vecinos (KNeighborsClassifier):** "vota" con los vecinos más cercanos.  
- **SVM (SVC):** maximiza el margen entre clases; kernels para no linealidad.  
- **Árboles (DecisionTreeClassifier):** reglas if-else; interpretables.  
- **Random Forest (RandomForestClassifier):** ensamble de árboles; robusto y suele generalizar bien.  
- **Naive Bayes:** asume independencia condicional; muy usado en texto.

> **Nota:** La elección depende de los **datos**, **recursos** y **métricas** objetivo.



## Flujo general de un clasificador (diagrama)
```
Datos crudos  →  Limpieza/EDA  →  Partición (train/test)
                                   ↓
                         Entrenamiento del modelo
                                   ↓
                               Predicción
                                   ↓
                              Evaluación
                                   ↓
                        Ajuste de hiperparámetros
                                   ↓
                               Despliegue
```


In [ ]:
# Preparación: librerías y utilidades
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, auc, ConfusionMatrixDisplay)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Configurar impresión y gráficos
np.set_printoptions(suppress=True)
pd.set_option("display.max_columns", 100)


## Vista rápida del dataset Iris

In [ ]:
# Ejemplo: Clasificación con Iris
# - Usaremos dos características para visualizar en 2D: Petal length (cm) y Petal width (cm)
# - Convertimos a un problema binario para incluir ROC (setosa vs no-setosa) y luego multiclase para comparar modelos

iris = load_iris(as_frame=True)
df = iris.frame.copy()
df.head()


## Entrenamiento y evaluación inicial

In [ ]:
# Seleccionamos dos características para la visualización 2D
X = df[["petal length (cm)", "petal width (cm)"]].values
y = iris.target.values  # multiclase: 0=setosa, 1=versicolor, 2=virginica

# División train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

# Nota sobre stratify
# stratify=y: Este argumento se utiliza para la división estratificada.
# Significa que la función dividirá los datos de manera que se preserve la
# proporción de cada clase en el conjunto de datos original (y).
# Esto es particularmente importante para los problemas de clasificación,
#especialmente cuando tienes clases desequilibradas, para asegurar que tanto
# los conjuntos de entrenamiento como los de prueba sean representativos de
# la distribución general de clases.

## El escalado de características utilizando StandardScaler.

### ¿Qué hace StandardScaler?

StandardScaler estandariza las características (variables de entrada) restando la media y dividiendo por la desviación estándar de cada característica. Esto da como resultado que cada característica tenga una media de 0 y una desviación estándar de 1. La fórmula es:

$$X_{scaled} = (X - mean) / std$$

¿Por qué es útil el escalado?

El escalado es importante para muchos algoritmos de aprendizaje automático porque son sensibles a la escala de las características. Aquí están las justificaciones clave:

- Algoritmos basados en distancia: Modelos como k-Nearest Neighbors (kNN) y Support Vector Machines (SVM) calculan distancias entre puntos de datos. Si las características tienen escalas muy diferentes (por ejemplo, una característica varía de 0 a 1000 y otra de 0 a 1), la característica con la escala más grande dominará el cálculo de la distancia, independientemente de su importancia real para la predicción. Escalar las características asegura que todas contribuyan por igual a la distancia.

- Algoritmos basados en gradiente: Algunos algoritmos (como la Regresión Logística y las redes neuronales) utilizan métodos de optimización basados en gradientes para encontrar los mejores parámetros del modelo. Si las características no están escaladas, el descenso de gradiente puede ser mucho más lento y difícil de converger, ya que el algoritmo tendrá que "navegar" por una superficie de pérdida más irregular.

### ¿Por qué árboles y bosques no lo requieren?

Los algoritmos basados en árboles (como Decision Trees y Random Forests) toman decisiones basadas en umbrales en características individuales (por ejemplo, "¿es la longitud del pétalo > 1.5 cm?"). Estas decisiones no se ven afectadas por la escala de las características, solo por sus valores relativos dentro de esa característica. Por lo tanto, el escalado no es necesario para estos modelos.

### ¿Por qué "no afecta" a árboles/forests?

Aunque no es necesario, aplicar escalado a las características antes de entrenar un árbol de decisión o un bosque aleatorio generalmente no perjudica su rendimiento. Simplemente no les proporciona ningún beneficio significativo en términos de convergencia o rendimiento del modelo.

In [ ]:
# Escalado (útil para SVM/kNN); los árboles/forests no lo requieren pero no afecta
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

X_train_scaled[:3], y_train[:10]


In [ ]:
# 4.1 Entrenamos un Árbol de Decisión como primer clasificador
tree = DecisionTreeClassifier(random_state=42, max_depth=3)
tree.fit(X_train, y_train)

y_pred_tree = tree.predict(X_test)
acc = accuracy_score(y_test, y_pred_tree)
prec = precision_score(y_test, y_pred_tree, average='macro')
rec = recall_score(y_test, y_pred_tree, average='macro')
f1 = f1_score(y_test, y_pred_tree, average='macro')

print(f"Accuracy: {acc:.3f} | Precision (macro): {prec:.3f} | Recall (macro): {rec:.3f} | F1 (macro): {f1:.3f}")


In [ ]:
# Matriz de confusión (multiclase) para el Árbol
fig, ax = plt.subplots(figsize=(5,4))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_tree, display_labels=iris.target_names, ax=ax)
ax.set_title("Matriz de Confusión - Árbol de Decisión")
plt.show()

print("Aciertos:", int((y_test == y_pred_tree).sum()), "de", len(y_test))


### Interpretación
Sume la diagonal de la matriz para contar aciertos; reste ese total al número de observaciones para obtener errores. Identifique las confusiones fuera de la diagonal y compruebe que los totales coinciden con el conjunto de prueba.

In [ ]:

# Visualización de la frontera de decisión en 2D (Árbol)
def plot_decision_regions(model, X, y, title, scaler=None, h=0.02):
    # si se pasa scaler, la malla se genera en espacio original pero se transforma para predecir
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    grid = np.c_[xx.ravel(), yy.ravel()]
    grid_scaled = scaler.transform(grid) if scaler is not None else grid
    Z = model.predict(grid_scaled if scaler is not None else grid)
    Z = Z.reshape(xx.shape)

    fig, ax = plt.subplots(figsize=(6,5))
    ax.contourf(xx, yy, Z, alpha=0.3)  # no especificar colores
    # puntos de entrenamiento
    for cls in np.unique(y):
        ax.scatter(X[y==cls,0], X[y==cls,1], label=f"Clase {cls}", edgecolor="k")
    ax.set_xlabel("Petal length (cm)")
    ax.set_ylabel("Petal width (cm)")
    ax.set_title(title)
    ax.legend()
    plt.show()

plot_decision_regions(tree, X_train, y_train, "Frontera de decisión (Árbol de Decisión)")


### La Curva ROC

Aquí tienes una explicación detallada de la Curva ROC (Receiver Operating Characteristic) que aparece en la imagen, correspondiente a la Regresión Logística (setosa vs no-setosa).

**¿Qué representa la Curva ROC?**

La Curva ROC muestra gráficamente la capacidad de un clasificador binario para distinguir entre las dos clases (positiva y negativa) a medida que varía el umbral de decisión (threshold).

En este caso:

*   Clase positiva (1) → setosa
*   Clase negativa (0) → no setosa (versicolor o virginica)

 **Ejes del gráfico**

*   **Eje X: Tasa de Falsos Positivos (FPR)**

$$ FPR = \frac{FP}{FP + TN} $$

    Cuanto más bajo, mejor (pocos falsos positivos).

*   **Eje Y: Tasa de Verdaderos Positivos (TPR o Recall)**

$$ TPR = \frac{TP}{TP + FN} $$

    Cuanto más alto, mejor (más positivos detectados correctamente).

 **Interpretación de la curva**

La curva traza puntos que representan distintos umbrales de probabilidad (por ejemplo, 0.1, 0.2, ..., 0.9). En cada umbral se calcula un par (FPR, TPR). El área bajo la curva (AUC) resume el rendimiento global.

*   AUC = 1.0 → Clasificador perfecto
*   AUC = 0.5 → Clasificador aleatorio (línea diagonal naranja)
*   AUC < 0.5 → Peor que el azar (modelo invertido)

 **En el presente gráfico:**

*   Curva azul → desempeño del modelo.
*   Línea naranja discontinua → referencia de azar (AUC=0.5).
*   AUC = 1.000 → significa que el modelo separa perfectamente las clases sin errores.

**Interpretación del resultado**

El modelo de Regresión Logística logra:

*   TPR = 1.0 y FPR = 0.0 en todos los puntos relevantes.
*   Clasifica setosa vs no-setosa sin confusiones.
*   AUC = 1.000, es decir, 100% de capacidad de discriminación.

Esto coincide con lo que vimos antes:

La clase setosa es perfectamente separable de las demás en el dataset Iris (sus pétalos son mucho más pequeños).

 **En resumen**

| Métrica      | Significado                                          | Interpretación                     |
| :----------- | :--------------------------------------------------- | :--------------------------------- |
| TPR (Recall) | Qué proporción de positivos reales fueron detectados | 1.0 → detectó todos los casos setosa |
| FPR          | Qué proporción de negativos fueron mal clasificados  | 0.0 → ningún no-setosa fue         |

In [ ]:

# 4.3 ROC-AUC (clasificación binaria: setosa vs no-setosa)
# Generamos etiquetas binarias: 1 si setosa, 0 en caso contrario
y_train_bin = (y_train == 0).astype(int)
y_test_bin  = (y_test == 0).astype(int)

logreg = LogisticRegression(random_state=42)
logreg.fit(X_train_scaled, y_train_bin)

# Probabilidad de clase positiva (setosa)
y_prob = logreg.predict_proba(X_test_scaled)[:,1]
fpr, tpr, thr = roc_curve(y_test_bin, y_prob)
roc_auc = auc(fpr, tpr)

print(f"ROC-AUC (LogisticRegression, setosa vs no-setosa): {roc_auc:.3f}")

# Curva ROC
fig, ax = plt.subplots(figsize=(5,4))
ax.plot(fpr, tpr, label=f"ROC curve (AUC = {roc_auc:.3f})")
ax.plot([0,1],[0,1], linestyle="--")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("Curva ROC - Regresión Logística (binaria)")
ax.legend()
plt.show()
